In [5]:
import pandas as pd
import ast

df = pd.read_csv('datasets/labeled_vulnerabilities_IoT.csv')
df['description_cleaned'] = df['description_cleaned'].apply(ast.literal_eval)
df.head()


C:\Users\kevin\AppData\Local\Temp\ipykernel_16204\2579005067.py:4: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('datasets/labeled_vulnerabilities_IoT.csv')


,id,description,datePublished,dateUpdated,baseScoreVersion,exploitedSince,baseScoreVector,epss,assigner,aliases,enisaIdVendor,references,enisaIdProduct,baseScore,enisaUuid,description_cleaned,product_name,vendor_name,product_label
0,EUVD-2026-24141,FreeScout is a free self-hosted help desk and ...,"Apr 21, 2026, 3:52:39 PM","Apr 21, 2026, 3:52:39 PM",3.1,NaN,CVSS:3.1/AV:N/AC:L/PR:N/UI:R/S:C/C:L/I:L/A:N,0.0,GitHub_M,CVE-2026-40565\n,[{'id': 'df8ea7fe-ea1b-3f31-a494-89b12dbf0c16'...,https://github.com/freescout-help-desk/freesco...,[{'id': '35cc8fe9-cc82-3bc7-9dda-d8dfd8a9d826'...,6.1,b3a6ee31-e20e-3b1c-a684-768095acc098,"[freescout, free, selfhosted, help, desk, shar...",freescout,freescout-help-desk,Other category
1,EUVD-2026-24138,Vulnerability related to an unquoted search pa...,"Apr 21, 2026, 3:32:22 PM","Apr 21, 2026, 3:32:22 PM",4.0,NaN,CVSS:4.0/AV:L/AC:L/AT:N/PR:L/UI:N/VC:H/VI:H/VA...,0.0,INCIBE,GHSA-9vxj-j2f7-9mgg\nCVE-2026-5789\n,[{'id': '5112208e-9449-3f30-b5a0-bdefa26b6740'...,https://www.incibe.es/en/incibe-cert/notices/a...,[{'id': '4dd00402-1cb8-33d1-91f8-5daf0e5fa5c5'...,8.5,7ed70757-c092-3a0b-99e2-96a7f815201b,"[vulnerability, related, unquoted, search, pat...",civetweb,CivetWeb,Other category
2,EUVD-2026-24136,"The method ""sock_recvfrom_into()"" of ""asyncio....","Apr 21, 2026, 3:32:22 PM","Apr 21, 2026, 3:32:22 PM",4.0,NaN,CVSS:4.0/AV:N/AC:L/AT:N/PR:N/UI:N/VC:L/VI:L/VA...,0.0,PSF,GHSA-3p9c-22jr-wq4x\nCVE-2026-3298\n,[{'id': '455eddad-0274-38e3-876c-1da485aedea5'...,https://github.com/python/cpython/pull/148809\...,[{'id': '79323943-f5b6-3a59-ac3e-e7849e5be63e'...,8.8,e0f49fce-f7e8-3dfc-b70e-dbc0c058a383,"[method, sock_recvfrom_into, asyncioproacterev...",CPython,Python Software Foundation,Other category
3,EUVD-2026-24130,User‑Controlled HTTP Header in Fortra's GoAnyw...,"Apr 21, 2026, 3:32:22 PM","Apr 21, 2026, 3:32:22 PM",3.1,NaN,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:L/I:N/A:L,0.0,Fortra,GHSA-6x5f-r479-qh4p\nCVE-2026-1089\n,[{'id': '2ae1e1ad-e03a-3855-b5d6-a2176c0b31dd'...,https://www.fortra.com/security/advisories/pro...,[{'id': '70404ca1-7e53-3be1-87b1-6dc09b1e4ff2'...,6.5,f1bfb170-4572-33bb-a02c-83c7a5e4ce08,"[usercontrolled, http, header, fortras, goanyw...",GoAnywhere MFT,Fortra,Other category
4,EUVD-2026-24129,The login limit is not enforced on the SFTP se...,"Apr 21, 2026, 3:32:22 PM","Apr 21, 2026, 3:32:22 PM",3.1,NaN,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:L/I:L/A:L,0.0,Fortra,GHSA-rpc6-m3h5-gmf2\nCVE-2026-0972\n,[{'id': 'cba00c9f-fa23-3062-a6a5-5709e85421d7'...,https://fortra.com/security/advisories/product...,[{'id': 'd104a637-0939-344c-be4c-448beede3920'...,7.3,3965ce35-f611-3fd6-bc4c-580538b77705,"[login, limit, enforced, sftp, service, fortra...",GoAnywhere MFT,Fortra,Other category


In [6]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary
import itertools

def get_top_words_per_topic(model, feature_names, n_words=10):
    topics = []
    for topic in model.components_:
        top_indices = topic.argsort()[:-n_words - 1:-1]
        topics.append([feature_names[i] for i in top_indices])
    return topics


def topic_diversity(topics):
    all_words = [w for topic in topics for w in topic]
    if not all_words:
        return 0.0
    return len(set(all_words)) / len(all_words)


def coherence_score(topics, tokenized_docs, dictionary):
    cm = CoherenceModel(
        topics=topics,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v',
    )
    return cm.get_coherence()

def print_topics(model, feature_names, n_words=10):
    for i, topic in enumerate(model.components_):
        top_words = [feature_names[j] for j in topic.argsort()[:-n_words - 1:-1]]
        print(f"Topic {i:02d}: {', '.join(top_words)}")


In [3]:
with open("lda_topics_all_IoT.txt", "a") as out:
    out.write("LDA Topic Results\n" + "="*60 + "\n\n")

to_check = ["Machinery"]
for category in to_check:
    print(f"Category: {category}")
    tokenized_docs = df[df["product_label"] == category]["description_cleaned"].tolist()

    if len(tokenized_docs) < 20:
        print(f"Skipping '{category}': only {len(tokenized_docs)} documents.")
        continue

    texts = [" ".join(doc) for doc in tokenized_docs]
    count_vectorizer = CountVectorizer(
        max_df=0.5,
        min_df=10,
        max_features=50000,
        ngram_range=(1, 2),
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9_\-]{2,}\b",
        stop_words='english',
    )
    dtm_counts = count_vectorizer.fit_transform(texts)
    count_features = count_vectorizer.get_feature_names_out()

    gensim_dict = Dictionary(tokenized_docs)

    k_range = [10, 15, 20, 25, 30, 35, 40]
    alpha_range = [0.05, 0.1, 0.3]
    beta_range = [None, 0.01, 0.1]  # None = sklearn auto

    lda_results = []

    for k, alpha, beta in itertools.product(k_range, alpha_range, beta_range):
        lda = LatentDirichletAllocation(
            n_components=k,
            doc_topic_prior=alpha,
            topic_word_prior=beta,
            max_iter=20,
            learning_method='batch',
            random_state=1,
        )
        lda.fit(dtm_counts)

        topics = get_top_words_per_topic(lda, count_features, n_words=10)
        coherence = coherence_score(topics, tokenized_docs, gensim_dict)
        diversity = topic_diversity(topics)

        lda_results.append({
            'k': k,
            'alpha': alpha,
            'beta': beta,
            'coherence': coherence,
            'diversity': diversity,
            'combined_score': coherence * diversity, #to get a score between 0 and 1, with both contributing equally
        })
        print(f"k={k:>2} alpha={alpha} beta={beta}: "
            f"coherence={coherence:.4f} diversity={diversity:.3f}")

    lda_results_df = pd.DataFrame(lda_results).sort_values('combined_score', ascending=False)

    best_config = lda_results_df.iloc[0]
    best_beta = None if pd.isna(best_config['beta']) else best_config['beta']
    final_model = LatentDirichletAllocation(
        n_components=int(best_config['k']),
        doc_topic_prior=int(best_config['alpha']),
        topic_word_prior=int(best_beta) if best_beta is not None else None,
        max_iter=20,
        learning_method='batch',
        random_state=1,
    )
    final_model.fit(dtm_counts)
    
    
    print(f"Best configuration for category '{category}': k={best_config['k']}, alpha={best_config['alpha']}, beta={best_config['beta']}, coherence={best_config['coherence']:.4f}, diversity={best_config['diversity']:.3f}, combined_score={best_config['combined_score']:.4f}")
    print_topics(final_model, count_features, n_words=10)
    
    with open("lda_topics_all_IoT.txt", "a") as out:
        out.write(f"Category: {category}\n")
        out.write(f"Best config: k={int(best_config['k'])}, alpha={best_config['alpha']}, beta={best_config['beta']}, "
                f"coherence={best_config['coherence']:.4f}, diversity={best_config['diversity']:.3f}, "
                f"combined_score={best_config['combined_score']:.4f}\n\n")
        for i, topic in enumerate(final_model.components_):
            top_words = [count_features[j] for j in topic.argsort()[:-11:-1]]
            out.write(f"Topic {i:02d}: {', '.join(top_words)}\n")
        out.write("\n" + "-"*60 + "\n\n")



Category: Machinery
k=10 alpha=0.05 beta=None: coherence=0.6643 diversity=0.670
k=10 alpha=0.05 beta=0.01: coherence=0.6840 diversity=0.710
k=10 alpha=0.05 beta=0.1: coherence=0.6643 diversity=0.670
k=10 alpha=0.1 beta=None: coherence=0.6643 diversity=0.690
k=10 alpha=0.1 beta=0.01: coherence=0.6577 diversity=0.710
k=10 alpha=0.1 beta=0.1: coherence=0.6643 diversity=0.690
k=10 alpha=0.3 beta=None: coherence=0.6893 diversity=0.760
k=10 alpha=0.3 beta=0.01: coherence=0.6874 diversity=0.750
k=10 alpha=0.3 beta=0.1: coherence=0.6893 diversity=0.760
k=15 alpha=0.05 beta=None: coherence=0.7441 diversity=0.713
k=15 alpha=0.05 beta=0.01: coherence=0.7479 diversity=0.713
k=15 alpha=0.05 beta=0.1: coherence=0.7519 diversity=0.700
k=15 alpha=0.1 beta=None: coherence=0.7312 diversity=0.653
k=15 alpha=0.1 beta=0.01: coherence=0.7301 diversity=0.687


KeyboardInterrupt: 

In [9]:
# For failed models manually
tokenized_docs_1 = df[df["product_label"] == "Machinery"]["description_cleaned"].tolist()
texts_1 = [" ".join(doc) for doc in tokenized_docs_1]

count_vectorizer_1 = CountVectorizer(
    max_df=0.5,
    min_df=10,
    max_features=50000,
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9_\-]{2,}\b",
    stop_words='english',
)
dtm_counts_1 = count_vectorizer_1.fit_transform(texts_1)
count_features_1 = count_vectorizer_1.get_feature_names_out()

final_model_1 = LatentDirichletAllocation(
    n_components=15,
    doc_topic_prior=0.05,
    topic_word_prior=0.01,
    max_iter=20,
    learning_method='batch',
    random_state=1,
)
final_model_1.fit(dtm_counts_1)
    
print_topics(final_model_1, count_features_1, n_words=10)




Topic 00: version, product, user, affected, read, allow, multiple, file, affected product, privilege
Topic 01: version, simatic, cpu, siplus, version simatic, version siplus, vulnerability identified, identified, device, affected
Topic 02: series, version, electric, prior, mitsubishi, mitsubishi electric, version prior, melsec, unauthenticated attacker, module
Topic 03: arbitrary, execute, code, execute arbitrary, remote, http, authenticated, user, crosssite, request
Topic 04: access, user, information, password, version, sensitive, credential, file, sensitive information, data
Topic 05: product, version, affected product, fasttools, affected, product version, plc, denialofservice, fault, electric
Topic 06: file, code, malicious, remote, parsing, execute, electric, result, fuji, fuji electric
Topic 07: issue, authentication, affect, issue affect, abb, bypass, gain, access, product, exists
Topic 08: file, process, lack, current, proper, current process, validation, execute, code, usersu

In [ ]:
# For failed models manually
tokenized_docs_1 = df[df["product_label"] == "PDF & Document Viewer/Editor Software"]["description_cleaned"].tolist()
texts_1 = [" ".join(doc) for doc in tokenized_docs_1]

count_vectorizer_1 = CountVectorizer(
    max_df=0.5,
    min_df=10,
    max_features=50000,
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9_\-]{2,}\b",
    stop_words='english',
)
dtm_counts_1 = count_vectorizer_1.fit_transform(texts_1)
count_features_1 = count_vectorizer_1.get_feature_names_out()

final_model_1 = LatentDirichletAllocation(
    n_components=10,
    doc_topic_prior=0.3,
    topic_word_prior=None,
    max_iter=20,
    learning_method='batch',
    random_state=1,
)
final_model_1.fit(dtm_counts_1)
    

print(f"Best configuration for category '{category}': k={best_config['k']}, alpha={best_config['alpha']}, beta={best_config['beta']}, coherence={best_config['coherence']:.4f}, diversity={best_config['diversity']:.3f}, combined_score={best_config['combined_score']:.4f}")
print_topics(final_model_1, count_features_1, n_words=10)

with open("lda_topics_all.txt", "a") as out:
    out.write(f"Category: {category}\n")
    out.write(f"Best config: k={int(best_config['k'])}, alpha={best_config['alpha']}, beta={best_config['beta']}, "
            f"coherence={best_config['coherence']:.4f}, diversity={best_config['diversity']:.3f}, "
            f"combined_score={best_config['combined_score']:.4f}\n\n")
    for i, topic in enumerate(final_model_1.components_):
        top_words = [count_features_1[j] for j in topic.argsort()[:-11:-1]]
        out.write(f"Topic {i:02d}: {', '.join(top_words)}\n")
    out.write("\n" + "-"*60 + "\n\n")



In [11]:
# For failed models manually
tokenized_docs_2 = df[df["product_label"] == "Web Server Software"]["description_cleaned"].tolist()
texts_2 = [" ".join(doc) for doc in tokenized_docs_2]

count_vectorizer_2 = CountVectorizer(
    max_df=0.5,
    min_df=10,
    max_features=50000,
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9_\-]{2,}\b",
    stop_words='english',
)
dtm_counts_2 = count_vectorizer_2.fit_transform(texts_2)
count_features_2 = count_vectorizer_2.get_feature_names_out()

final_model_2 = LatentDirichletAllocation(
    n_components=15,
    doc_topic_prior=0.1,
    topic_word_prior=None,
    max_iter=20,
    learning_method='batch',
    random_state=1,
)
final_model_2.fit(dtm_counts_2)
    

print(f"Best configuration for category '{category}': k={best_config['k']}, alpha={best_config['alpha']}, beta={best_config['beta']}, coherence={best_config['coherence']:.4f}, diversity={best_config['diversity']:.3f}, combined_score={best_config['combined_score']:.4f}")
print_topics(final_model_2, count_features_2, n_words=10)

with open("lda_topics_all.txt", "a") as out:
    out.write(f"Category: {category}\n")
    out.write(f"Best config: k={int(best_config['k'])}, alpha={best_config['alpha']}, beta={best_config['beta']}, "
            f"coherence={best_config['coherence']:.4f}, diversity={best_config['diversity']:.3f}, "
            f"combined_score={best_config['combined_score']:.4f}\n\n")
    for i, topic in enumerate(final_model_2.components_):
        top_words = [count_features_2[j] for j in topic.argsort()[:-11:-1]]
        out.write(f"Topic {i:02d}: {', '.join(top_words)}\n")
    out.write("\n" + "-"*60 + "\n\n")



Best configuration for category 'Legal, Compliance & Privacy Management Software': k=10.0, alpha=0.1, beta=nan, coherence=0.7060, diversity=0.800, combined_score=0.5648
Topic 00: request, header, http, puma, proxy, response, fixed, http request, client, attacker
Topic 01: access, allows, web, affect, web server, attacker, issue, issue affect, cache, application
Topic 02: apache, http, http server, apache http, issue, fix, upgrade, user, recommended, user recommended
Topic 03: attacker, remote, allows, code, arbitrary, execute, privilege, service, malicious, remote attacker
Topic 04: file, path, directory, web, web server, traversal, path traversal, allows, information, patch
Topic 05: user, allow, send, remote, remote user, allow remote, crafted, specially crafted, specially, affecting
Topic 06: connection, service, denial service, denial, request, thread, remote, web, web server, attacker
Topic 07: command, injection, code, command injection, improper, arbitrary, allows, control, code

In [12]:
# For failed models manually
tokenized_docs_3 = df[df["product_label"] == "Wearable & Consumer Electronics"]["description_cleaned"].tolist()
texts_3 = [" ".join(doc) for doc in tokenized_docs_3]

count_vectorizer_3 = CountVectorizer(
    max_df=0.5,
    min_df=10,
    max_features=50000,
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9_\-]{2,}\b",
    stop_words='english',
)
dtm_counts_3 = count_vectorizer_3.fit_transform(texts_3)
count_features_3 = count_vectorizer_3.get_feature_names_out()

final_model_3 = LatentDirichletAllocation(
    n_components=30,
    doc_topic_prior=0.1,
    topic_word_prior=None,
    max_iter=20,
    learning_method='batch',
    random_state=1,
)
final_model_3.fit(dtm_counts_3)
    

print(f"Best configuration for category '{category}': k={best_config['k']}, alpha={best_config['alpha']}, beta={best_config['beta']}, coherence={best_config['coherence']:.4f}, diversity={best_config['diversity']:.3f}, combined_score={best_config['combined_score']:.4f}")
print_topics(final_model_3, count_features_3, n_words=10)

with open("lda_topics_all.txt", "a") as out:
    out.write(f"Category: {category}\n")
    out.write(f"Best config: k={int(best_config['k'])}, alpha={best_config['alpha']}, beta={best_config['beta']}, "
            f"coherence={best_config['coherence']:.4f}, diversity={best_config['diversity']:.3f}, "
            f"combined_score={best_config['combined_score']:.4f}\n\n")
    for i, topic in enumerate(final_model_3.components_):
        top_words = [count_features_3[j] for j in topic.argsort()[:-11:-1]]
        out.write(f"Topic {i:02d}: {', '.join(top_words)}\n")
    out.write("\n" + "-"*60 + "\n\n")



Best configuration for category 'Legal, Compliance & Privacy Management Software': k=10.0, alpha=0.1, beta=nan, coherence=0.7060, diversity=0.800, combined_score=0.5648
Topic 00: firmware, update, http, firmware update, server, exploitable, request, device, vulnerability exists, exists
Topic 01: attack, lead, component, manipulation, unknown, manipulation lead, vendor, disclosure, contacted, early
Topic 02: service, allows attacker, improper, denial, denial service, smr oct2021, oct2021, oct2021 release, mar2022, mar2022 release
Topic 03: apr2022 release, apr2022, smr apr2022, library, library prior, function, bound, libsimba library, libsimba, remote attacker
Topic 04: information, sensitive, sensitive information, local attacker, local, access, allows local, attacker access, exposure, exposure sensitive
Topic 05: execute, code, specific, result, sonos, vulnerability allows, exploit, data, exploit vulnerability, flaw
Topic 06: access, control, access control, improper access, improper